# WordPiece from scratch — step by step
We train using pair scores and split text into Unicode words and punctuation. This is a simplified WordPiece trainer, not a reproduction of Google's original trainer or full BERT preprocessing.

Flow: corpus → word counts → character pieces → pair scores → merges → vocabulary → longest-match encoding → IDs → JSON.

References: [WordPiece training lesson](https://huggingface.co/learn/llm-course/en/chapter6/6)

## Block 1 — Imports and settings

The `##` prefix means a piece continues a preceding word. `[UNK]` represents a whole word we cannot split. Keep the vocabulary target small so each training step is easy to inspect.

In [1]:
from collections import Counter
from fractions import Fraction
from pathlib import Path
import json
import re
import sys
import tempfile

PREFIX = '##'
UNK = '[UNK]'
SPECIAL_TOKENS = ['[PAD]', UNK, '[CLS]', '[SEP]', '[MASK]']
TARGET_VOCAB_SIZE = 30
MAX_WORD_CHARS = 100
LOWERCASE = False

print('Python kernel:', sys.executable)

Python kernel: d:\project\.venv\Scripts\python.exe


## Block 2 — A small corpus and pre-tokenization

This corpus repeats words intentionally: their frequencies affect training. Punctuation is split into its own token. Lowercasing, if enabled, must be identical during training and inference.

In [2]:
corpus = ['hug hug hug hug hug hug hug hug hug hug',
          'pug pug pug pug pug',
          'pun pun pun pun pun pun pun pun pun pun pun pun',
          'bun bun bun bun',
          'hugs hugs hugs hugs hugs']


def pre_tokenize(text):
    if LOWERCASE:
        text = text.lower()
    return re.findall(r'\w+|[^\w\s]', text, flags=re.UNICODE)


print(pre_tokenize('Hug, pug!'))
# Expected: ['Hug', ',', 'pug', '!']

['Hug', ',', 'pug', '!']


## Block 3 — Count words

Store unique words and occurrence counts, not a copy of every occurrence. Expected counts: `hug: 10`, `pug: 5`, `pun: 12`, `bun: 4`, `hugs: 5`.

In [3]:
word_counts = Counter()
for text in corpus:
    word_counts.update(pre_tokenize(text))

print(word_counts)
assert word_counts == Counter({'hug': 10, 'pug': 5, 'pun': 12, 'bun': 4, 'hugs': 5})

Counter({'pun': 12, 'hug': 10, 'pug': 5, 'hugs': 5, 'bun': 4})


## Block 4 — Start with character pieces

The first character is unprefixed. Every later character is prefixed.

```text
hugs → h ##u ##g ##s
```

The vocabulary retains these base pieces even after they participate in merges.

In [4]:
splits = Counter()
for word, frequency in word_counts.items():
    pieces = (word[0],) + tuple(PREFIX + char for char in word[1:])
    splits[pieces] += frequency

alphabet = {piece for pieces in splits for piece in pieces}
vocabulary = SPECIAL_TOKENS + sorted(alphabet - set(SPECIAL_TOKENS))

for pieces, frequency in splits.items():
    print(pieces, 'frequency:', frequency)
print('Initial vocabulary:', vocabulary)

('h', '##u', '##g') frequency: 10
('p', '##u', '##g') frequency: 5
('p', '##u', '##n') frequency: 12
('b', '##u', '##n') frequency: 4
('h', '##u', '##g', '##s') frequency: 5
Initial vocabulary: ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '##g', '##n', '##s', '##u', 'b', 'h', 'p']


## Block 5 — Count pieces and adjacent pairs

Count every occurrence, weighted by its word frequency. Repeated pieces within one word must be counted more than once. For example, `##u` occurs 36 times in this corpus.

In [5]:
def count_pieces_and_pairs(word_splits):
    piece_counts = Counter()
    pair_counts = Counter()
    for pieces, frequency in word_splits.items():
        for piece in pieces:
            piece_counts[piece] += frequency
        for pair in zip(pieces, pieces[1:]):
            pair_counts[pair] += frequency
    return piece_counts, pair_counts


piece_counts, pair_counts = count_pieces_and_pairs(splits)
print('Pieces:', piece_counts)
print('Pairs:', pair_counts)
assert piece_counts['##u'] == 36

Pieces: Counter({'##u': 36, '##g': 20, 'p': 17, '##n': 16, 'h': 15, '##s': 5, 'b': 4})
Pairs: Counter({('##u', '##g'): 20, ('p', '##u'): 17, ('##u', '##n'): 16, ('h', '##u'): 15, ('##g', '##s'): 5, ('b', '##u'): 4})


## Block 6 — Calculate WordPiece pair scores

```text
score = pair_count / (first_piece_count × second_piece_count)
```

Example: `(##g, ##s)` has count 5. `##g` occurs 20 times and `##s` occurs 5 times, so its score is `5 / (20 × 5) = 1/20`.

Using `Fraction` avoids floating-point differences. Lexical ordering breaks ties reproducibly.

In [6]:
def get_pair_scores(word_splits):
    piece_counts, pair_counts = count_pieces_and_pairs(word_splits)
    return {pair: Fraction(count, piece_counts[pair[0]] * piece_counts[pair[1]])
            for pair, count in pair_counts.items()}


scores = get_pair_scores(splits)
for pair, score in sorted(scores.items(), key=lambda item: (-item[1], item[0])):
    print(pair, 'score:', score)

best_pair = min(scores, key=lambda pair: (-scores[pair], pair))
print('Best pair:', best_pair)
assert best_pair == ('##g', '##s')

('##g', '##s') score: 1/20
('##u', '##g') score: 1/36
('##u', '##n') score: 1/36
('b', '##u') score: 1/36
('h', '##u') score: 1/36
('p', '##u') score: 1/36
Best pair: ('##g', '##s')


## Block 7 — Merge one pair

Remove the continuation prefix from the second piece only:

```text
##g + ##s → ##gs
h + ##u   → hu
```

Merge left to right without overlapping replacements, then carry word frequencies forward.

In [7]:
def merge_pair(word_splits, pair):
    merged_piece = pair[0] + pair[1][len(PREFIX):]
    updated = Counter()
    for pieces, frequency in word_splits.items():
        result = []
        index = 0
        while index < len(pieces):
            if pieces[index:index + 2] == pair:
                result.append(merged_piece)
                index += 2
            else:
                result.append(pieces[index])
                index += 1
        updated[tuple(result)] += frequency
    return updated, merged_piece


merged_splits, new_piece = merge_pair(splits, best_pair)
print('Added piece:', new_piece)
for pieces, frequency in merged_splits.items():
    print(pieces, frequency)
assert new_piece == '##gs'

Added piece: ##gs
('h', '##u', '##g') 10
('p', '##u', '##g') 5
('p', '##u', '##n') 12
('b', '##u', '##n') 4
('h', '##u', '##gs') 5


## Block 8 — Repeat to build the vocabulary

Start from the original splits so this cell can be rerun. Stop when the target vocabulary size is reached, or when there are no more pairs. Retain every learned piece, including those absorbed into later pieces. WordPiece encoding needs the final vocabulary—not the training merge order.

In [8]:
if TARGET_VOCAB_SIZE < len(SPECIAL_TOKENS) + len(alphabet):
    raise ValueError('Target vocabulary must fit the initial alphabet and special tokens')

training_splits = splits.copy()
vocabulary = SPECIAL_TOKENS + sorted(alphabet - set(SPECIAL_TOKENS))
known = set(vocabulary)
training_trace = []

while len(vocabulary) < TARGET_VOCAB_SIZE:
    scores = get_pair_scores(training_splits)
    if not scores:
        break
    best = min(scores, key=lambda pair: (-scores[pair], pair))
    training_splits, new_piece = merge_pair(training_splits, best)
    if new_piece not in known:
        vocabulary.append(new_piece)
        known.add(new_piece)
    training_trace.append((best, new_piece))
    print(best, '→', new_piece)

print('Final vocabulary size:', len(vocabulary))
print(vocabulary)

('##g', '##s') → ##gs
('##u', '##g') → ##ug
('##u', '##gs') → ##ugs
('h', '##ugs') → hugs
('h', '##ug') → hug
('##u', '##n') → ##un
('b', '##un') → bun
('p', '##ug') → pug
('p', '##un') → pun
Final vocabulary size: 21
['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', '##g', '##n', '##s', '##u', 'b', 'h', 'p', '##gs', '##ug', '##ugs', 'hugs', 'hug', '##un', 'bun', 'pug', 'pun']


## Block 9 — Greedy longest-match tokenization

Try the longest vocabulary piece beginning at the current position. Later pieces require `##`. If any position cannot be matched, discard partial pieces and return one `[UNK]` for the entire word.

This is intentionally greedy: it does not backtrack to try shorter earlier pieces.

In [9]:
def tokenize_word(word, vocabulary_set):
    if LOWERCASE:
        word = word.lower()
    if len(word) > MAX_WORD_CHARS:
        return [UNK]
    result = []
    start = 0
    while start < len(word):
        end = len(word)
        found = None
        while end > start:
            candidate = word[start:end]
            if start:
                candidate = PREFIX + candidate
            if candidate in vocabulary_set:
                found = candidate
                break
            end -= 1
        if found is None:
            return [UNK]
        result.append(found)
        start = end
    return result


manual_vocab = {UNK, 'play', 'p', '##lay', '##ing'}
print(tokenize_word('playing', manual_vocab))  # ['play', '##ing']
print(tokenize_word('playz', manual_vocab))    # ['[UNK]'], not ['play', '[UNK]']
assert tokenize_word('playing', manual_vocab) == ['play', '##ing']
assert tokenize_word('playz', manual_vocab) == [UNK]

['play', '##ing']
['[UNK]']


## Block 10 — Encode a sentence and assign token IDs

Token IDs are positions in the ordered vocabulary. Preserve that order in the saved model. IDs for special tokens are reserved, but this example does not automatically insert `[CLS]` or `[SEP]`.

In [10]:
def tokenize_text(text, vocabulary_set):
    return [piece for word in pre_tokenize(text)
            for piece in tokenize_word(word, vocabulary_set)]


token_to_id = {piece: index for index, piece in enumerate(vocabulary)}
example = 'hugs bugs mug'
tokens = tokenize_text(example, set(vocabulary))
ids = [token_to_id[piece] for piece in tokens]
print('Input:', example)
print('Pieces:', tokens)
print('IDs:', ids)
assert UNK in tokens  # 'mug' cannot start with a known 'm'.

Input: hugs bugs mug
Pieces: ['hugs', 'b', '##ugs', '[UNK]']
IDs: [15, 9, 14, 1]


## Block 11 — Decode

Join continuation pieces onto their preceding word. Decoding creates canonical spaces; it cannot restore original spacing, case lost by lowercasing, or text replaced by `[UNK]`.

In [11]:
def decode_tokens(tokens):
    words = []
    for token in tokens:
        if token.startswith(PREFIX):
            if not words:
                raise ValueError('Continuation piece has no preceding word')
            words[-1] += token[len(PREFIX):]
        else:
            words.append(token)
    return ' '.join(words)


print(decode_tokens(['play', '##ing', 'walk', '##ing']))
print(decode_tokens([vocabulary[index] for index in ids]))
assert decode_tokens(['play', '##ing']) == 'playing'

playing walking
hugs bugs [UNK]


## Block 12 — Save and reload JSON

Store the vocabulary **and** settings. The example uses a temporary directory so rerunning it never overwrites your project models. The file is deleted automatically after the block finishes.

In [12]:
model = {
    'type': 'WordPiece', 'version': 1,
    'vocabulary': vocabulary, 'lowercase': LOWERCASE,
    'prefix': PREFIX, 'unk_token': UNK, 'max_chars': MAX_WORD_CHARS,
    'pre_tokenization': 'unicode_words_and_punctuation',
}

with tempfile.TemporaryDirectory() as folder:
    model_path = Path(folder) / 'wordpiece_model.json'
    with model_path.open('x', encoding='utf-8') as file:
        json.dump(model, file, ensure_ascii=False, indent=2)
    with model_path.open(encoding='utf-8') as file:
        restored = json.load(file)
    assert restored['vocabulary'] == vocabulary
    restored_ids = {piece: index for index, piece in enumerate(restored['vocabulary'])}
    assert restored_ids == token_to_id
    print('JSON round-trip verified:', model_path)

JSON round-trip verified: C:\Users\tiwar\AppData\Local\Temp\tmpa8m52qcb\wordpiece_model.json


## Block 13 — Compare with the modular implementation

Now use `WordPiece/code` to confirm that your scratch training and encoding behave the same way. This cell locates the project root whether the kernel starts in the project folder or inside `WordPiece`.

In [13]:
project_root = next((path for path in [Path.cwd(), *Path.cwd().parents]
                     if (path / 'WordPiece' / 'code' / 'training.py').is_file()), None)
if project_root is None:
    raise RuntimeError('Start the kernel inside tokenizer_implementations or its WordPiece folder')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from WordPiece.code import WordPieceTokenizer, train_wordpiece

package_vocab = train_wordpiece(corpus, TARGET_VOCAB_SIZE, LOWERCASE,
                               PREFIX, SPECIAL_TOKENS)
assert package_vocab == vocabulary
package_tokenizer = WordPieceTokenizer(package_vocab, LOWERCASE, PREFIX, UNK, MAX_WORD_CHARS)
assert package_tokenizer.tokenize(example) == tokens
assert package_tokenizer.encode(example) == ids
print('Scratch code matches the modular implementation!')

Scratch code matches the modular implementation!
